<a href="https://colab.research.google.com/github/TERK93/sensor-pulse/blob/main/sensor_pulse_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sensor-pulse — Industrial Sensor Pipeline
**Dataset:** NASA CMAPSS Turbofan Engine Degradation  
**Stack:** PySpark · Spark SQL · Parquet · Medallion Architecture

This notebook implements an end-to-end data pipeline for industrial sensor data,
following the Medallion Architecture (Bronze → Silver → Gold).

The NASA CMAPSS dataset simulates turbofan engine sensor readings over time —
21 sensors per engine cycle measuring temperature, pressure, and rotation speed.
This mirrors industrial IoT data from production environments like breweries,
manufacturing plants, and process industries.

| Layer  | Purpose |
|--------|---------|
| Bronze | Raw ingestion → Parquet, append-only |
| Silver | Validation, status flags, derived columns |
| Gold   | Analytics views + ML feature engineering |

## 1. Setup

In [ ]:
!pip install pyspark kagglehub -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.window import Window
import kagglehub
import shutil
import os

spark = SparkSession.builder \
    .appName("SensorPulse") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.0.2


## 2. Download Data

In [ ]:
path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
print("Downloaded to:", path)

os.makedirs("data/raw", exist_ok=True)
files = ["train_FD001.txt", "train_FD002.txt", "train_FD003.txt", "train_FD004.txt"]
for f in files:
    shutil.copy(f"{path}/CMaps/{f}", f"data/raw/{f}")
    print(f"File ready: data/raw/{f}")

Using Colab cache for faster access to the 'nasa-cmaps' dataset.
Downloaded to: /kaggle/input/nasa-cmaps
File ready: data/raw/train_FD001.txt
File ready: data/raw/train_FD002.txt
File ready: data/raw/train_FD003.txt
File ready: data/raw/train_FD004.txt


## 3. Schema

CMAPSS columns: engine_id, cycle, 3 operational settings, 21 sensor readings.  
Defining schema explicitly avoids type inference errors — good practice for production pipelines.

In [ ]:
sensor_cols = [StructField(f"sensor_{i:02d}", DoubleType(), True) for i in range(1, 22)]

schema = StructType([
    StructField("engine_id",    IntegerType(), True),
    StructField("cycle",        IntegerType(), True),
    StructField("op_setting_1", DoubleType(),  True),
    StructField("op_setting_2", DoubleType(),  True),
    StructField("op_setting_3", DoubleType(),  True),
] + sensor_cols)

print(f"Schema defined: {len(schema.fields)} columns")

Schema defined: 26 columns


## 4. Bronze Layer — Raw Ingestion

Bronze = append-only, no transformations. Store exactly what was received.
`load_timestamp` added for data lineage tracking.

In [ ]:
from functools import reduce

dfs = []
for dataset_name in ["FD001", "FD002", "FD003", "FD004"]:
    df = spark.read \
        .schema(schema) \
        .option("sep", " ") \
        .option("header", "false") \
        .csv(f"data/raw/train_{dataset_name}.txt") \
        .withColumn("dataset", F.lit(dataset_name))
    dfs.append(df)

df_raw = reduce(lambda a, b: a.union(b), dfs)
df_bronze = df_raw.withColumn("load_timestamp", F.current_timestamp())

print(f"Rows ingested:  {df_bronze.count():,}")
print(f"Engines:        {df_bronze.select('engine_id').distinct().count()}")
df_bronze.groupBy("dataset").count().orderBy("dataset").show()

Rows ingested:  160,359
Engines:        260
+-------+-----+
|dataset|count|
+-------+-----+
|  FD001|20631|
|  FD002|53759|
|  FD003|24720|
|  FD004|61249|
+-------+-----+



In [ ]:
os.makedirs("data/bronze", exist_ok=True)
df_bronze.write.mode("overwrite").parquet("data/bronze/sensor_readings")
print("Bronze written to data/bronze/sensor_readings")

Bronze written to data/bronze/sensor_readings


## 5. Silver Layer — Validation & Cleaning

Validation rules:
1. **Null check** — nulls in engine_id, cycle or key sensors → `invalid_null`
2. **Sensor range** — sensor values must be positive → `invalid_sensor_range`

Design decision: invalid rows are **kept with a status flag**, not deleted.
Data lineage is preserved. Gold filters on `WHERE status = 'valid'`.

In [ ]:
df_bronze = spark.read.parquet("data/bronze/sensor_readings")

key_sensors = ["sensor_02", "sensor_03", "sensor_04", "sensor_07", "sensor_11"]

null_condition = F.lit(False)
for col in ["engine_id", "cycle"] + key_sensors:
    null_condition = null_condition | F.col(col).isNull()

negative_condition = F.lit(False)
for col in key_sensors:
    negative_condition = negative_condition | (F.col(col) <= 0)

window_engine = Window.partitionBy("dataset", "engine_id")

df_silver = df_bronze \
    .withColumn(
        "status",
        F.when(null_condition,     F.lit("invalid_null"))
         .when(negative_condition, F.lit("invalid_sensor_range"))
         .otherwise(               F.lit("valid"))
    ) \
    .withColumn("max_cycle",        F.max("cycle").over(window_engine)) \
    .withColumn("cycle_pct",        F.round(F.col("cycle") / F.col("max_cycle"), 3)) \
    .withColumn("silver_timestamp", F.current_timestamp())

print("=== Validation Summary ===")
df_silver.groupBy("status").count().show()

df_silver.select("engine_id", "cycle", "max_cycle", "cycle_pct", "sensor_02", "sensor_04", "status").show(5)

=== Validation Summary ===
+------+------+
|status| count|
+------+------+
| valid|160359|
+------+------+

+---------+-----+---------+---------+---------+---------+------+
|engine_id|cycle|max_cycle|cycle_pct|sensor_02|sensor_04|status|
+---------+-----+---------+---------+---------+---------+------+
|        1|    1|      192|    0.005|   641.82|   1400.6| valid|
|        1|    2|      192|     0.01|   642.15|  1403.14| valid|
|        1|    3|      192|    0.016|   642.35|   1404.2| valid|
|        1|    4|      192|    0.021|   642.35|  1401.87| valid|
|        1|    5|      192|    0.026|   642.37|  1406.22| valid|
+---------+-----+---------+---------+---------+---------+------+
only showing top 5 rows


In [ ]:
os.makedirs("data/silver", exist_ok=True)
df_silver.write.mode("overwrite").parquet("data/silver/sensor_readings_validated")
print(f"Silver written — {df_silver.filter(F.col('status') == 'valid').count():,} valid rows")

Silver written — 160,359 valid rows


## 6. Gold Layer — Spark SQL Analytics

Gold = analytics-ready views built on validated Silver data.  
Registering Silver as a Spark SQL temp view — the same pattern used in Microsoft Fabric and Databricks.

In [ ]:
df_silver = spark.read.parquet("data/silver/sensor_readings_validated")
df_valid = df_silver.filter(F.col("status") == "valid")
df_valid.createOrReplaceTempView("silver_sensors")

print(f"Valid rows registered as SQL view: {df_valid.count():,}")

Valid rows registered as SQL view: 160,359


### Gold View 1 — Rolling Sensor Averages
Rolling averages smooth noise and reveal degradation trends over time.

In [ ]:
gold_rolling = spark.sql("""
    SELECT
        engine_id,
        cycle,
        sensor_02,
        sensor_04,
        sensor_11,
        ROUND(AVG(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 3) AS sensor_02_avg_10,
        ROUND(AVG(sensor_04) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 3) AS sensor_04_avg_10,
        ROUND(AVG(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 3) AS sensor_02_avg_30
    FROM silver_sensors
    ORDER BY engine_id, cycle
""")

gold_rolling.createOrReplaceTempView("gold_rolling_averages")
print(f"gold_rolling_averages: {gold_rolling.count():,} rows")
gold_rolling.show(5)

gold_rolling_averages: 160,359 rows
+---------+-----+---------+---------+---------+----------------+----------------+----------------+
|engine_id|cycle|sensor_02|sensor_04|sensor_11|sensor_02_avg_10|sensor_04_avg_10|sensor_02_avg_30|
+---------+-----+---------+---------+---------+----------------+----------------+----------------+
|        1|    1|   641.82|   1400.6|    47.47|          641.82|          1400.6|          641.82|
|        1|    1|   555.32|  1137.23|    42.02|          598.57|        1268.915|          598.57|
|        1|    1|   642.36|  1396.84|     47.3|         613.167|        1311.557|         613.167|
|        1|    1|   549.68|  1112.93|    41.69|         597.295|          1261.9|         597.295|
|        1|    2|   642.15|  1403.14|    47.49|         606.266|        1290.148|         606.266|
+---------+-----+---------+---------+---------+----------------+----------------+----------------+
only showing top 5 rows


### Gold View 2 — Engine Health Summary

In [ ]:
gold_health = spark.sql("""
    SELECT
        engine_id,
        MAX(cycle)               AS total_cycles,
        ROUND(AVG(sensor_02), 2) AS avg_sensor_02,
        ROUND(AVG(sensor_04), 2) AS avg_sensor_04,
        ROUND(AVG(sensor_11), 2) AS avg_sensor_11,
        ROUND(STDDEV(sensor_02), 3) AS stddev_sensor_02,
        CASE
            WHEN MAX(cycle) > 200 THEN 'long_life'
            WHEN MAX(cycle) > 130 THEN 'medium_life'
            ELSE 'short_life'
        END AS life_category
    FROM silver_sensors
    GROUP BY engine_id
    ORDER BY total_cycles DESC
""")

gold_health.createOrReplaceTempView("gold_engine_health")
print("Life category distribution:")
gold_health.groupBy("life_category").count().show()
gold_health.show(10)

Life category distribution:
+-------------+-----+
|life_category|count|
+-------------+-----+
|    long_life|  225|
|  medium_life|   35|
+-------------+-----+

+---------+------------+-------------+-------------+-------------+----------------+-------------+
|engine_id|total_cycles|avg_sensor_02|avg_sensor_04|avg_sensor_11|stddev_sensor_02|life_category|
+---------+------------+-------------+-------------+-------------+----------------+-------------+
|      118|         543|       579.94|       1202.3|        42.86|          37.742|    long_life|
|       55|         525|       616.72|      1320.52|        45.49|          39.121|    long_life|
|       24|         494|       619.19|      1329.17|        45.71|          37.786|    long_life|
|       96|         491|       618.93|      1327.85|         45.7|          38.206|    long_life|
|      133|         489|       580.15|      1200.64|        42.81|          37.344|    long_life|
|       10|         481|       614.25|      1310.11|   

### Gold View 3 — ML Feature Engineering

Features prepared for a Remaining Useful Life (RUL) prediction model:
- Rolling mean and stddev (signal stability)
- Rate of change via LAG (is sensor trending up or down?)
- RUL as target variable: `max_cycle - cycle`

This is the bridge between data engineering and machine learning.

In [ ]:
gold_features = spark.sql("""
    SELECT
        engine_id,
        cycle,
        cycle_pct,
        sensor_02,
        sensor_04,
        sensor_11,
        ROUND(AVG(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 3) AS s02_mean_10,
        ROUND(STDDEV(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 4) AS s02_std_10,
        ROUND(sensor_02 - LAG(sensor_02, 1) OVER (
            PARTITION BY engine_id ORDER BY cycle
        ), 4) AS s02_delta,
        ROUND(sensor_04 - LAG(sensor_04, 1) OVER (
            PARTITION BY engine_id ORDER BY cycle
        ), 4) AS s04_delta,
        (max_cycle - cycle) AS rul
    FROM silver_sensors
    ORDER BY engine_id, cycle
""")

gold_features.createOrReplaceTempView("gold_ml_features")
print(f"gold_ml_features: {gold_features.count():,} rows")
gold_features.show(5)

# Sanity check: RUL should decrease as cycle increases
print("\nRUL sanity check (engine 1):")
spark.sql("SELECT engine_id, cycle, rul FROM gold_ml_features WHERE engine_id = 1 ORDER BY cycle LIMIT 5").show()

gold_ml_features: 160,359 rows
+---------+-----+---------+---------+---------+---------+-----------+----------+---------+---------+---+
|engine_id|cycle|cycle_pct|sensor_02|sensor_04|sensor_11|s02_mean_10|s02_std_10|s02_delta|s04_delta|rul|
+---------+-----+---------+---------+---------+---------+-----------+----------+---------+---------+---+
|        1|    1|    0.005|   641.82|   1400.6|    47.47|     641.82|      NULL|     NULL|     NULL|191|
|        1|    1|    0.007|   555.32|  1137.23|    42.02|     598.57|   61.1647|    -86.5|  -263.37|148|
|        1|    1|    0.004|   642.36|  1396.84|     47.3|    613.167|   50.0974|    87.04|   259.61|258|
|        1|    1|    0.003|   549.68|  1112.93|    41.69|    597.295|   51.7765|   -92.68|  -283.91|320|
|        1|    2|     0.01|   642.15|  1403.14|    47.49|    606.266|   49.1223|    92.47|   290.21|190|
+---------+-----+---------+---------+---------+---------+-----------+----------+---------+---------+---+
only showing top 5 rows


### Gold View 4 — Sensor Drift (Early vs Late Life)

Compare sensor readings in first 30 cycles vs last 30 cycles per engine.
Positive drift = sensor value increased toward end of life → indicates degradation.

In [ ]:
gold_drift = spark.sql("""
    WITH early AS (
        SELECT engine_id,
            AVG(sensor_02) AS s02_early,
            AVG(sensor_04) AS s04_early,
            AVG(sensor_11) AS s11_early
        FROM silver_sensors
        WHERE cycle <= 30
        GROUP BY engine_id
    ),
    late AS (
        SELECT engine_id,
            AVG(sensor_02) AS s02_late,
            AVG(sensor_04) AS s04_late,
            AVG(sensor_11) AS s11_late
        FROM silver_sensors
        WHERE (max_cycle - cycle) <= 30
        GROUP BY engine_id
    )
    SELECT
        e.engine_id,
        ROUND(l.s02_late - e.s02_early, 3) AS s02_drift,
        ROUND(l.s04_late - e.s04_early, 3) AS s04_drift,
        ROUND(l.s11_late - e.s11_early, 3) AS s11_drift
    FROM early e
    JOIN late l ON e.engine_id = l.engine_id
    ORDER BY ABS(s02_drift) DESC
""")

gold_drift.createOrReplaceTempView("gold_sensor_drift")
print("Top 10 engines by sensor drift:")
gold_drift.show(10)

Top 10 engines by sensor drift:
+---------+---------+---------+---------+
|engine_id|s02_drift|s04_drift|s11_drift|
+---------+---------+---------+---------+
|      156|   18.223|   75.786|    2.202|
|      203|   16.483|   66.396|    1.832|
|      196|   15.791|   61.856|    1.284|
|      160|   15.438|   67.909|    1.944|
|      185|   15.395|   59.951|    1.743|
|      250|  -14.971|  -33.513|   -0.881|
|      257|   14.403|   64.741|    2.214|
|       90|   13.757|   59.026|    1.534|
|      180|   13.523|   60.298|    1.577|
|      177|  -13.245|   -28.46|   -0.872|
+---------+---------+---------+---------+
only showing top 10 rows


## 7. Write Gold Layer

In [ ]:
os.makedirs("data/gold", exist_ok=True)

gold_rolling.write.mode("overwrite").parquet("data/gold/rolling_averages")
gold_health.write.mode("overwrite").parquet("data/gold/engine_health")
gold_features.write.mode("overwrite").parquet("data/gold/ml_features")
gold_drift.write.mode("overwrite").parquet("data/gold/sensor_drift")

print("All Gold tables written:")
print("  data/gold/rolling_averages")
print("  data/gold/engine_health")
print("  data/gold/ml_features      <- input for anomaly detection notebook")
print("  data/gold/sensor_drift")

All Gold tables written:
  data/gold/rolling_averages
  data/gold/engine_health
  data/gold/ml_features      <- input for anomaly detection notebook
  data/gold/sensor_drift


# Summary
print("""
| Layer  | Format  | Rows    | Notes                                    |
|--------|---------|---------|------------------------------------------|
| Bronze | Parquet | 160,359 | Raw + load_timestamp + dataset tag       |
| Silver | Parquet | 160,359 | Validated, status-flagged, cycle_pct     |
| Gold   | Parquet | 4 tables| Rolling avgs, health, ML features, drift |

Datasets: FD001 (20,631) · FD002 (53,759) · FD003 (24,720) · FD004 (61,249)
Next: Anomaly detection notebook
""")
